In [ ]:
import time
from photonic_testing import *
MODBUS_PORT='/dev/tty.usbserial-B003T6PZ'
MODBUS_PORT='/dev/tty.usbserial-B003T6RF'
LASER_ADDRESSES = {"1028": 5,
                   "1270": 6,
                   "yj1430": 3,
                   "hk1430": 4,
                   "1510": 1,
                   "2330": 2}

LASER_DRIVER_SERIALS = {"1028": 8229,
                   "1270": 8228,
                   "yj1430": 8227,
                   "hk1430": 8222,
                   "1510": 8225,
                   "2330": 8226}

### Overview

See `ait/photonic_testing.py` for `Laser` and `LaserProperties` along with the specific limits of the individual laser diodes.


### Low-level access

In the event that direct device control is needed.

See `ait/maiman_modbus/utils/utils.py` for python constants of register names and `ait/maiman_modbus/config/modbus_config.yaml` for the register addresses.

Look at methods on `ModbusDevice` (`ait/maiman_modbus/device/modbus_device.py`) for functions.


```python
from maiman_modbus.communication import ModbusCommunication
import maiman_modbus.utils as maiman_regs
from maiman_modbus.device.modbus_device_model import ModbusDeviceModel
from maiman_modbus.config import DeviceConfig
from maiman_modbus.device.modbus_device import ModbusDevice

d = ModbusDevice(port=MODBUS_PORT, slave_address=modbus_address)
d.comm.send_command(d.model.get_register(STATE_OF_TEC_COMMAND), MODBUS_START_TEC_COMMAND_VALUE)

print(d.comm.receive_response(d.model.get_register(STATE_OF_TEC_COMMAND)))
```


## Initialize all the diodes

Running this cell will create the `lasers` dictionary with a `Laser` for each laser.

In [ ]:
name = ('yj1430', )
names = tuple(LASER_ADDRESSES.keys())
lasers = {}
for name in names:
    l = Laser(name, address=LASER_ADDRESSES[name], MODBUS_PORT=MODBUS_PORT)
    serial = l.device.get_serial_number()
    print(f'🆔 Serial number: {serial}')
    assert l.device.get_serial_number()==LASER_DRIVER_SERIALS[name], 'BAD BUS CONFIG, do not continue'
    lasers[name] = l
    print('')


for name in names:
    serial = lasers[name].device.get_serial_number()
    assert serial==LASER_DRIVER_SERIALS[name], 'BAD BUS CONFIG, do not continue'

### Start the TECs

In [ ]:
for name in names:
    lasers[name].disable_interlock_and_cool()

Look at the status of one of them. It seems that the TEC status isn't polling well, but the temp changes.

In [ ]:
l = lasers['1270']
l.status()
tec_current = l.comm.receive_response(l.model.get_register('tec_current_measured'))
tec_voltage = l.comm.receive_response(l.model.get_register('tec_voltage'))
print(f'TEC Current: {tec_current} Voltage: {tec_voltage} V')

### Turn on a laser

This sets a percentage between the maximum current and the threshold current.

In [ ]:
lasers['1270'].set_current_as_percent(.5)

### Make sure it is all off.

In [ ]:
for name in names:
    lasers[name].shutdown()